In [ ]:
!git clone https://github.com/douglas-wesley/RC-fusion-architecture.git /content/fusion
%cd /content/fusion
!ls -la

In [ ]:
# Célula 2:
import os
from google.colab import drive

# 1. Monta o Drive
drive.mount('/content/drive')

# 2. Pasta que for colocar no seu drive
DIR = "PUT_HERE"

# 3. Caminhos das pastas
checkpoints_dir = f'/content/drive/MyDrive/{DIR}/checkpoints'
logs_dir = f'/content/drive/MyDrive/{DIR}/logs'

# 4. Validação e criação
for directory in [checkpoints_dir, logs_dir]:
    if os.path.exists(directory):
        print(f" Pasta já existe: {directory}")
    else:
        os.makedirs(directory, exist_ok=True)
        print(f" Pasta criada com sucesso: {directory}")


In [ ]:
# ─── Célula 3: Instalação Limpa e Estabilização de Dependências ───
# Essa base é necessária para evitar conflitos com relação as dependências do próprio Colab (numpy, pandas, etc).
# Ele provávelmente retornara algum erro e irá reiniciar a sessão, a partir disso, deve-se executar A PARTIR DA CÉLULA 4 em diante!!
import sys

print("1. Instalando versões compatíveis do ecossistema nuScenes...")
# Travamos o NumPy no 1.26.4 e o Shapely no 2.0.3 (exigências estritas do nuscenes-devkit)
!pip install -q "numpy==1.26.4" "shapely==2.0.3" parameterized

print("2. Instalando nuscenes-devkit e dependências do projeto...")
!pip install -q nuscenes-devkit einops pyyaml tqdm matplotlib tensorboard pycocotools

print("\n" + "="*60)
import numpy as np
print(f"Versão detectada do NumPy: {np.__version__}")

if np.__version__ != "1.26.4":
    print("ATENÇÃO: O Colab precisa reiniciar o kernel para aplicar o NumPy 1.26.4.")
    print("Reiniciando o ambiente automaticamente em 3 segundos...")
    import os, time
    time.sleep(2)
    os.kill(os.getpid(), 9)  # Força o Colab a recarregar o kernel limpo
else:
    print("[TUDO PRONTO] O ambiente está 100% compatível! Prossiga para a Célula 4.")
print("="*60)

In [ ]:
import os

os.makedirs('/content/data/sets', exist_ok=True)

if not os.path.exists('/content/data/sets/nuscenes/v1.0-mini'):
    print("Baixando nuScenes mini (~1.5 GB)...")
    !wget -q --show-progress -O /content/v1.0-mini.tgz https://www.nuscenes.org/data/v1.0-mini.tgz
    print("Extraindo arquivos...")
    !tar -xzf /content/v1.0-mini.tgz -C /content/data/sets/
    !rm /content/v1.0-mini.tgz
    print("Dataset extraído com sucesso!")
else:
    print("Dataset já existe.")

In [ ]:
# Cria a pasta nuscenes e move os arquivos para dentro dela caso tenham ficado soltos
!mkdir -p /content/data/sets/nuscenes

# Se as pastas foram extraídas soltas em /content/data/sets/:
!mv /content/data/sets/v1.0-mini /content/data/sets/nuscenes/ 2>/dev/null || true
!mv /content/data/sets/samples /content/data/sets/nuscenes/ 2>/dev/null || true
!mv /content/data/sets/sweeps /content/data/sets/nuscenes/ 2>/dev/null || true
!mv /content/data/sets/maps /content/data/sets/nuscenes/ 2>/dev/null || true

# Conferir se agora /content/data/sets/nuscenes/v1.0-mini existe:
!ls -la /content/data/sets/nuscenes/v1.0-mini


In [ ]:
# Uploada os checkpoints
!mkdir -p /content/fusion/checkpoints
!cp /content/drive/MyDrive/PIBIC/checkpoints/best_model.pth /content/fusion/checkpoints/

In [ ]:
# Célula de Treinamento:
%cd /content/fusion/
!python -m src.engine.train \
    --config config/default.yaml \
    --dataroot /content/data/sets/nuscenes


In [ ]:
# Célula de Evaluation:
%cd /content/fusion
!python -m src.engine.evaluate \
    --config config/default.yaml \
    --checkpoint checkpoints/best_model.pth \
    --dataroot /content/data/sets/nuscenes


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from src.engine.train import load_config, build_model, load_checkpoint
from src.dataset.nuscenes_dataset import NuScenesFusionDataset
from src.dataset.radar_transforms import radar_to_bev_grid

cfg = load_config("config/default.yaml")
cfg["data"]["dataroot"] = "/content/data/sets/nuscenes"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = build_model(cfg, device)
load_checkpoint(model, None, "checkpoints/best_model.pth", device)
model.eval()

val_dataset = NuScenesFusionDataset(
    dataroot="/content/data/sets/nuscenes",
    split="val",
    image_size=tuple(cfg["data"]["image_size"]),
    bev_size=tuple(cfg["data"]["bev_size"]),
    bev_range=tuple(cfg["data"]["bev_range"]),
)

# Escolha o índice da cena (ex: 5, 12, 20, 35)
sample_idx = 10

sample = val_dataset[sample_idx]
image = sample["image"].unsqueeze(0).to(device)
radar_bev = radar_to_bev_grid(
    sample["radar_points"].to(device),
    sample["radar_mask"].to(device),
    bev_size=tuple(cfg["data"]["bev_size"]),
    bev_range=tuple(cfg["data"]["bev_range"]),
).unsqueeze(0)

with torch.no_grad():
    seg_logits = model(image, radar_bev)
    prob = torch.sigmoid(seg_logits).cpu().squeeze().numpy()

img_show = sample["image"].permute(1, 2, 0).numpy()
img_show = np.clip(img_show * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406], 0, 1)
gt = sample["bev_segmentation"].squeeze().numpy()
radar_density = radar_bev[0, 0].cpu().numpy()

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(img_show)
axes[0].set_title(f"Amostra #{sample_idx} - Câmera RGB")
axes[0].axis("off")

axes[1].imshow(radar_density, cmap="viridis")
axes[1].set_title("Radar BEV (Densidade)")
axes[1].axis("off")

axes[2].imshow(gt, cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Ground Truth BEV")
axes[2].axis("off")

axes[3].imshow(prob > 0.4, cmap="magma", vmin=0, vmax=1)
axes[3].set_title("Predição (Limiar 0.4)")
axes[3].axis("off")

plt.tight_layout()
plt.show()
